In [ ]:
import torch, torchvision
import torch.nn as nn
import torch.nn.functional as F
import wandb
import logging


# 定义和工具函数
class EncoderMLP(nn.Module):
    def __init__(self):
        super().__init__()

class DecoderMLP(nn.Module):
    def __init__(self):
        super().__init__()

class Transformer(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size

        self.q = nn.Linear(input_size, hidden_size)
        self.k = nn.Linear(input_size, hidden_size)
        self.v = nn.Linear(input_size, hidden_size)

    def forward(self, x):
        q = self.q(x) # (seq_size, hidden)
        k = self.k(x) # (seq_size, hidden)
        v = self.v(x) # (seq_size, hidden)
        # score里面的每个元素都是hidden个normal distribution multi再add
        score = torch.matmul(q, k.transpose(-1, -2)) # (seq_size, seq_size)
        score = F.softmax(score / torch.sqrt(torch.tensor(self.hidden_size)), dim=-1) # (seq_size, seq_size)
        out = score @ v
        # 后面咋写啊？我记得还有残差啥的，位置编码我也没有加好像
        return out

class DecoderTransformer(nn.Module):
    def __init__(self, gauss_size, token_size, hidden_size, output_size, num_layers=1):
        super().__init__()
        self.gauss_size = gauss_size
        self.token_size = token_size
        self.transformers = nn.ModuleList([Transformer(hidden_size, hidden_size, output_size) for _ in range(num_layers)])
        self.embedding = nn.Embedding(10, token_size)

        self.gauss_linear = nn.Linear(gauss_size, hidden_size)
        self.token_linear = nn.Linear(token_size, hidden_size)

        self.out_linear = nn.Linear(hidden_size, output_size)

    def forward(self, num: torch.Tensor, gauss_noise: torch.Tensor):
        # num: (*)
        # gauss_noise: (*, gauss_size)
        num = num.unsqueeze(dim=-1) # (*, 1)
        num = num.to(torch.int)
        token_tensor = self.embedding(num) # (*, 1, embed_size)
        gauss_tensor = gauss_noise.unsqueeze(dim=-2) # (*, 1, gauss_size)
        token_tensor = self.token_linear(token_tensor)
        gauss_tensor = self.gauss_linear(gauss_tensor)

        x = torch.cat((gauss_tensor, token_tensor), dim=-2)

        for transformer in self.transformers:
            x = transformer(x)
        x = self.out_linear(x)

        return torch.sigmoid(x)

HYPERPARAMS = {
    "lr": 1e-4,
    "batch_size": 200,
    "epochs": 40,
    "token_size": 16,
}

